# C07-AE-03 — El mismo enmascaramiento y RLS dentro de un Snowflake Notebook

Este notebook corre con el rol **FR_BI_ANALYST** dentro de su propia sesión (el dueño del notebook es `FR_DEMO_ADMIN`, que tiene ese rol concedido). No hay código de seguridad en el notebook: el enmascaramiento de `RUT` y `NUMERO_TARJETA` y el filtro por unidad de negocio los aplica Snowflake sobre la tabla `CREDIT_CARD_TRANSACTIONS`, igual que en la consulta SQL de la sesión 3.

Resultados esperados (verificados en la cuenta, 2026-10-29): RUT como seudónimo `***-**-<hash>`, tarjeta `****-****-****-****`, solo RETAIL (15 969 797 filas).

In [ ]:
# 1. Identidad: el notebook empieza como FR_DEMO_ADMIN; cambiamos a FR_BI_ANALYST
print('Rol del notebook (owner):', session.sql('SELECT CURRENT_ROLE()').collect()[0][0])
session.use_role('FR_BI_ANALYST')
session.sql('USE SECONDARY ROLES NONE').collect()   # solo FR_BI_ANALYST activo
session.sql('USE WAREHOUSE WH_APP_MEDIUM').collect()
print('Rol activo:', session.sql('SELECT CURRENT_ROLE(), CURRENT_SECONDARY_ROLES()').collect()[0])

## 2. Enmascaramiento de columnas
RUT y número de tarjeta llegan enmascarados al analista. Las mismas columnas, sin cambios en el código.

In [ ]:
session.sql('''
SELECT RUT, NUMERO_TARJETA, CATEGORIA, MONTO, ESTADO, UNIDAD_NEGOCIO
FROM SUPERINTENDENCY_DEMO_DB.CORE_BANKING_SCHEMA.CREDIT_CARD_TRANSACTIONS
ORDER BY MONTO DESC
LIMIT 5
''').show()

## 3. Seguridad por fila (RLS)
El analista solo ve la unidad de negocio RETAIL. Las otras unidades no aparecen: no hay filas que filtrar en la aplicación.

In [ ]:
session.sql('''
SELECT UNIDAD_NEGOCIO, COUNT(*) AS TRANSACCIONES
FROM SUPERINTENDENCY_DEMO_DB.CORE_BANKING_SCHEMA.CREDIT_CARD_TRANSACTIONS
GROUP BY UNIDAD_NEGOCIO
ORDER BY UNIDAD_NEGOCIO
''').show()

## 4. El seudónimo es determinístico
Filtrar por el RUT enmascarado que el analista ya ve devuelve filas. Por eso los joins y filtros siguen funcionando con datos protegidos.

In [ ]:
token = session.sql('SELECT MIN(RUT) FROM SUPERINTENDENCY_DEMO_DB.CORE_BANKING_SCHEMA.CREDIT_CARD_TRANSACTIONS').collect()[0][0]
filas = session.sql(
    'SELECT COUNT(*) FROM SUPERINTENDENCY_DEMO_DB.CORE_BANKING_SCHEMA.CREDIT_CARD_TRANSACTIONS WHERE RUT = ?',
    params=[token],
).collect()[0][0]
print('Seudónimo:', token, '| filas con ese mismo seudónimo:', filas)

## 5. Opcional: contraste con el rol administrador
Solo para comparar. `FR_DEMO_ADMIN` ve el dato en claro. Vuelve a `FR_BI_ANALYST` al final.

In [ ]:
session.use_role('FR_DEMO_ADMIN')
session.sql('SELECT RUT, NUMERO_TARJETA, UNIDAD_NEGOCIO FROM SUPERINTENDENCY_DEMO_DB.CORE_BANKING_SCHEMA.CREDIT_CARD_TRANSACTIONS LIMIT 3').show()
session.sql('SELECT UNIDAD_NEGOCIO, COUNT(*) AS TRANSACCIONES FROM SUPERINTENDENCY_DEMO_DB.CORE_BANKING_SCHEMA.CREDIT_CARD_TRANSACTIONS GROUP BY UNIDAD_NEGOCIO ORDER BY UNIDAD_NEGOCIO').show()
session.use_role('FR_BI_ANALYST')
print('De vuelta como:', session.sql('SELECT CURRENT_ROLE()').collect()[0][0])